### CELDA 1

In [1]:
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuración de rutas
DATA_DIR = Path("data/crudos_por_cohorte")
OUTPUT_TSV = "espectro_ALL.tsv"

print(f"Directorio de datos: {DATA_DIR.resolve()}")
print(f"Existe: {DATA_DIR.exists()}")

Directorio de datos: /Users/daniel/Documents/INMEGEN/AUTOENCODER_V3/data/crudos_por_cohorte
Existe: True


### CELDA 2

In [2]:
# Buscar todos los archivos que inicien con "espectro_"
archivos = sorted(DATA_DIR.glob("espectro_*.tsv"))

print(f"Archivos encontrados: {len(archivos)}")
for a in archivos[:5]:
    print(f"  - {a.name}")
if len(archivos) > 5:
    print(f"  ... y {len(archivos) - 5} más")

# Leer y concatenar
dfs = []
for archivo in archivos:
    try:
        df_temp = pd.read_csv(archivo, sep="\t")
        df_temp["source_file"] = archivo.name  # opcional: trazabilidad
        dfs.append(df_temp)
    except Exception as e:
        print(f"⚠️ Error leyendo {archivo.name}: {e}")

espectro_all = pd.concat(dfs, ignore_index=True)

# Guardar el consolidado
espectro_all.to_csv(OUTPUT_TSV, sep="\t", index=False)

print(f"\n✅ Consolidado guardado en: {OUTPUT_TSV}")
print(f"Shape: {espectro_all.shape}")
print(f"Columnas: {list(espectro_all.columns)}")
espectro_all.head()

Archivos encontrados: 5
  - espectro_GSE130970.tsv
  - espectro_GSE135251.tsv
  - espectro_GSE142530.tsv
  - espectro_GSE162694.tsv
  - espectro_GSE276114.tsv

✅ Consolidado guardado en: espectro_ALL.tsv
Shape: (5045090, 12)
Columnas: ['chr', 'k', 'phase', 'N', 'power', 'sample', 'condition', 'window_power', 'n_observed', 'n_dropped_unmeasured', 'coverage', 'source_file']


,chr,k,phase,N,power,sample,condition,window_power,n_observed,n_dropped_unmeasured,coverage,source_file
0,1,1,2.690075,2066,0.007087,GSM3758035,F0,0.001150,1418,0,0.68635,espectro_GSE130970.tsv
1,1,2,-0.111537,2066,0.000011,GSM3758035,F0,0.002289,1418,0,0.68635,espectro_GSE130970.tsv
2,1,3,-1.682474,2066,0.002019,GSM3758035,F0,0.001771,1418,0,0.68635,espectro_GSE130970.tsv
3,1,4,3.112548,2066,0.012183,GSM3758035,F0,0.000113,1418,0,0.68635,espectro_GSE130970.tsv
4,1,5,1.362093,2066,0.007337,GSM3758035,F0,0.000150,1418,0,0.68635,espectro_GSE130970.tsv


### CELDA 3

In [3]:
conteo_sample = (
    espectro_all["sample"]
    .value_counts()
    .rename_axis("sample")
    .reset_index(name="n_registros")
    .sort_values("n_registros", ascending=False)
    .reset_index(drop=True)
)

print(f"Total de samples únicos: {conteo_sample.shape[0]}")
print(f"Total de registros: {conteo_sample['n_registros'].sum()}")
print("\nResumen estadístico de registros por sample:")
print(conteo_sample["n_registros"].describe())

conteo_sample

Total de samples únicos: 503
Total de registros: 5045090

Resumen estadístico de registros por sample:
count      503.0
mean     10030.0
std          0.0
min      10030.0
25%      10030.0
50%      10030.0
75%      10030.0
max      10030.0
Name: n_registros, dtype: float64


,sample,n_registros
0,GSM3758035,10030
1,GSM3758010,10030
2,GSM3758039,10030
3,GSM3758041,10030
4,GSM3758042,10030
...,...,...
498,Liver sample 114,10030
499,Liver sample 115,10030
500,Liver sample 116,10030
501,Liver sample 117,10030


### CELDA 4

In [11]:
# ============================================================
# 4. Tablas comparativas de CHR por SAMPLE
# ============================================================

# ---------- 4.1 Frecuencia: cuántas veces aparece cada chr en cada sample ----------
chr_freq = (
    espectro_all.groupby(["sample", "chr"])
    .size()
    .rename("n_registros")
    .reset_index()
)

# ---------- 4.2 Tabla 1: Sample × CHR (conteos) ----------
tabla_chr = chr_freq.pivot_table(
    index="sample",
    columns="chr",
    values="n_registros",
    fill_value=0,
    aggfunc="sum",
)

# Orden natural de columnas (numéricas primero, luego strings)
def _key(c):
    c = str(c)
    return (0, int(c)) if c.isdigit() else (1, c)

tabla_chr = tabla_chr.reindex(sorted(tabla_chr.columns, key=_key), axis=1)

# Exportar Tabla 1
TABLA_CHR_TSV = "reporte_tabla_chr_por_sample.tsv"
tabla_chr.to_csv(TABLA_CHR_TSV, sep="\t")

print(f"✅ Tabla 1 guardada: {TABLA_CHR_TSV}")
print(f"   Shape: {tabla_chr.shape[0]} samples × {tabla_chr.shape[1]} chr")
tabla_chr.head()




# ============================================================
# 4b. Tabla 2: solo los chr que difieren entre samples
# ============================================================

# Un chr se queda si, a lo largo de TODOS los samples,
# no todos los conteos son idénticos.
chr_diferentes = [
    col for col in tabla_chr.columns
    if tabla_chr[col].nunique() > 1
]

tabla_chr_dif = tabla_chr[chr_diferentes].copy()

# Exportar Tabla 2
TABLA_DIF_TSV = "reporte_chr_diferencias.tsv"
tabla_chr_dif.to_csv(TABLA_DIF_TSV, sep="\t")

print(f"✅ Tabla 2 guardada: {TABLA_DIF_TSV}")
print(f"   Chr totales:        {tabla_chr.shape[1]}")
print(f"   Chr con diferencias: {len(chr_diferentes)}")
print(f"   Chr idénticos en todos los samples: {tabla_chr.shape[1] - len(chr_diferentes)}")
print(f"   Chr con diferencias: {chr_diferentes}")

tabla_chr_dif.head()

✅ Tabla 1 guardada: reporte_tabla_chr_por_sample.tsv
   Shape: 503 samples × 24 chr
✅ Tabla 2 guardada: reporte_chr_diferencias.tsv
   Chr totales:        24
   Chr con diferencias: 0
   Chr idénticos en todos los samples: 24
   Chr con diferencias: []


chr
sample
Control_091119
Control_100088
Control_100630
Control_100733
Control_111089


### CELDA 5

In [6]:
# Normalizar chr a string para evitar mezcla de int/str
espectro_all["chr"] = espectro_all["chr"].astype(str)

# Comparar cada sample contra el "chr esperado"
chr_global = set(espectro_all["chr"].unique())
print(f"Chr presentes en TODO el dataset: {len(chr_global)}")

# Orden "natural": primero los que son numéricos, luego el resto
def _key(c):
    return (0, int(c)) if c.isdigit() else (1, c)

chr_sorted = sorted(chr_global, key=_key)
print(f"Chr: {chr_sorted}\n")  # mejor mostrar TODOS para inspección

# Chr por sample como sets
chr_sets = espectro_all.groupby("sample")["chr"].apply(lambda x: set(x.unique()))

# Detectar samples incompletos
reporte_homogeneidad = pd.DataFrame({
    "sample": chr_sets.index,
    "n_chr": chr_sets.apply(len),
    "chr_faltantes": chr_sets.apply(lambda s: sorted(chr_global - s, key=_key)),
    "n_chr_faltantes": chr_sets.apply(lambda s: len(chr_global - s)),
}).reset_index(drop=True)

reporte_homogeneidad["completo"] = reporte_homogeneidad["n_chr_faltantes"] == 0
reporte_homogeneidad = reporte_homogeneidad.sort_values(
    ["n_chr_faltantes", "sample"], ascending=[False, True]
).reset_index(drop=True)

print(f"Samples completos (todos los chr): {reporte_homogeneidad['completo'].sum()} / {len(reporte_homogeneidad)}")
print(f"Samples con chr faltantes: {(~reporte_homogeneidad['completo']).sum()}\n")

reporte_homogeneidad

Chr presentes en TODO el dataset: 24
Chr: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', 'X', 'Y']

Samples completos (todos los chr): 503 / 503
Samples con chr faltantes: 0



,sample,n_chr,chr_faltantes,n_chr_faltantes,completo
0,Control_091119,24,[],0,True
1,Control_100088,24,[],0,True
2,Control_100630,24,[],0,True
3,Control_100733,24,[],0,True
4,Control_111089,24,[],0,True
...,...,...,...,...,...
498,Liver sample 91,24,[],0,True
499,Liver sample 96,24,[],0,True
500,Liver sample 97,24,[],0,True
501,Liver sample 98,24,[],0,True


### CELDA 6

In [7]:
problematicos = reporte_homogeneidad[~reporte_homogeneidad["completo"]]
if problematicos.empty:
    print("✅ Todos los samples tienen el mismo conjunto de chr.")
else:
    print(f"⚠️ {len(problematicos)} samples con chr faltantes:\n")
    display(problematicos.head(30))

# Distribución de cuántos chr tiene cada sample
print("\nDistribución de n_chr por sample:")
print(reporte_homogeneidad["n_chr"].value_counts().sort_index())

✅ Todos los samples tienen el mismo conjunto de chr.

Distribución de n_chr por sample:
n_chr
24    503
Name: count, dtype: int64


### CELDA 7

In [8]:
conteo_sample.to_csv("reporte_conteo_sample.tsv", sep="\t", index=False)
tabla_chr.to_csv("reporte_tabla_chr_por_sample.tsv", sep="\t")
reporte_homogeneidad.to_csv("reporte_homogeneidad.tsv", sep="\t", index=False)

print("✅ Reportes exportados:")
print("  - reporte_conteo_sample.tsv")
print("  - reporte_tabla_chr_por_sample.tsv")
print("  - reporte_homogeneidad.tsv")

✅ Reportes exportados:
  - reporte_conteo_sample.tsv
  - reporte_tabla_chr_por_sample.tsv
  - reporte_homogeneidad.tsv
